# Lab 09: BERT Fine-tuning

**Module 09** | Read `notes/09-bert.pdf` before running this notebook.

Fine-tune DistilBERT on SST-2 sentiment with the Hugging Face Trainer API and report validation accuracy.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## DistilBERT fine-tuning on SST-2

BERT-style encoders produce contextual embeddings for every token. For classification we attach a small head on the `[CLS]` representation and fine-tune the whole stack on labeled sentences. **DistilBERT** retains most of the accuracy of BERT-base with roughly half the parameters, which makes it a practical default for instructional fine-tuning.

SST-2 is a binary sentiment benchmark (positive vs negative movie reviews). We load `SetFit/sst2`, a drop-in compatible with the classic GLUE version, and use the Hugging Face `Trainer` API so training, evaluation, and metric logging stay declarative.


The tokenizer converts sentences to `input_ids` and an attention mask. Sequences are padded and truncated to 128 tokens. Training runs for **one epoch** with batch size 8; we report validation accuracy after `trainer.evaluate()`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from runtime_env import configure_runtime

configure_runtime()

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 8

raw = load_dataset("SetFit/sst2")
print(raw)
print("Train example:", raw["train"][0])


Tokenization maps each split to tensors the model consumes. Labels are already integers (`0` = negative, `1` = positive). We drop raw text columns after encoding to keep batches lean.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch: dict) -> dict:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

encoded = raw.map(tokenize_batch, batched=True)
encoded = encoded.rename_column("label", "labels")
encoded.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
print(f"Model loaded: {MODEL_NAME}")


`TrainingArguments` sets one epoch, batch size 8, and evaluation after training. `Trainer` handles the optimization loop, device placement, and metric aggregation, the same entry point used in production fine-tuning pipelines.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {"accuracy": float(accuracy)}


training_args = TrainingArguments(
    output_dir=str(ROOT / "labs" / "outputs" / "distilbert_sst2"),
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
metrics = trainer.evaluate()
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Validation accuracy: {metrics['eval_accuracy']:.4f}")
